In [168]:
# 판다스 넘파이 가져오기
import pandas as pd
import numpy as np

In [169]:
# 파일 불러오기
order = pd.read_csv('./data/orders.csv')
oi = pd.read_csv('./data/order_items.csv')
prod = pd.read_csv('./data/products.csv')
cust = pd.read_csv('./data/customers.csv')

In [170]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [171]:
# orders 파일 정제
print(f'주문일시 결측 주문 건수 : {order["order_datetime"].isna().sum()}')
orders=order.dropna(subset='order_datetime')
print(f'주문일시 결측 제거 후 주문 건수 : {orders.shape[0]}')
orders=orders.drop_duplicates(subset='order_id')
print(f'중복 제거 후 주문건 수 : {orders.shape[0]}')
orders['order_datetime']=pd.to_datetime(orders['order_datetime'],errors="coerce")
orders.info()

주문일시 결측 주문 건수 : 1933
주문일시 결측 제거 후 주문 건수 : 198067
중복 제거 후 주문건 수 : 198067
<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 11.2 MB


In [172]:
#product함수 관련 결측치 및 제거
prod=prod.drop_duplicates()
prod['price']=pd.to_numeric(prod['price'].astype(str)
                            .str.replace(',','').str.replace('원',''))
prod= prod[prod['price']>0]
prod= prod[prod['cost']>0]
prod['category']=prod['category'].str.strip()
prod.info()

<class 'pandas.DataFrame'>
Index: 489 entries, 1 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    489 non-null    int64  
 1   product_name  489 non-null    str    
 2   category      489 non-null    str    
 3   price         489 non-null    float64
 4   cost          489 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 35.1 KB


In [173]:
# order_item 파일 정제 (1, 주문 아이템 아이디 중복제거)
orderitem=oi.drop_duplicates(subset='order_item_id')
orderitem.info()

<class 'pandas.DataFrame'>
Index: 499880 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     484774 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 26.7 MB


In [174]:
# order_item 파일 정제 (2. 유닛 프라이스 채우기)
orderitem= orderitem.merge(prod[['product_id','price']],how='left',on='product_id')
orderitem['unit_price']= orderitem['unit_price'].fillna(orderitem["price"])

orderitem=orderitem.drop_duplicates()
orderitem=orderitem.drop(columns='price')
orderitem.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     499641 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [175]:
# customer 파일 정제
cust.info()
cust.head()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  5000 non-null   int64
 1   name         5000 non-null   str  
 2   gender       4696 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  5000 non-null   str  
 5   city         5000 non-null   str  
 6   email        4752 non-null   str  
dtypes: int64(1), str(6)
memory usage: 544.3 KB


,customer_id,name,gender,birth_date,signup_date,city,email
0,3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com
1,1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com
2,4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com
3,4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com
4,3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com


In [176]:
#중복치 제거
cust=cust.drop_duplicates(subset='customer_id')
#birth_date, signup_date 정제
cust['birth_date']=pd.to_datetime(cust['birth_date'],errors="coerce")
cust['signup_date']=pd.to_datetime(cust['signup_date'],errors="coerce")
# city 정제
cust['city']=cust['city'].str.strip()
#gender 정제
gmap={'여':'F','F':'F','남':'M','M':'M'}
cust['gender']=cust['gender'].map(gmap)
#결측치는 빼지 않고, 일단 냅두기로 함. 데이터에 사용가치가 있음.

In [177]:
cust.info()

<class 'pandas.DataFrame'>
Index: 4950 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customer_id  4950 non-null   int64         
 1   name         4950 non-null   str           
 2   gender       4649 non-null   str           
 3   birth_date   4572 non-null   datetime64[us]
 4   signup_date  4950 non-null   datetime64[us]
 5   city         4950 non-null   str           
 6   email        4702 non-null   str           
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 479.4 KB


In [178]:
# orders & order_items 결합
mix_ooi = orderitem.merge(orders,how='left',on='order_id',validate="m:1")
mix_ooi.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   499880 non-null  int64         
 1   order_id        499880 non-null  int64         
 2   product_id      499880 non-null  int64         
 3   quantity        499880 non-null  int64         
 4   unit_price      499641 non-null  float64       
 5   discount        499880 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(4), str(2)
memory usage: 43.5 MB


In [179]:
#주문정보가 존재하지 않는, 주문 품목 갯수 발견, 정확한 주문과 주문일시를 확인이 불가능하므로,
# 제거 예정
print(f'주문에 들어가있지 않은 주문 품목 수 : {mix_ooi["order_datetime"].isna().sum()}')
mix_ooi = mix_ooi.dropna(subset='order_datetime')
print(f'결측치 제거 후 총 주문품목수 : {mix_ooi.shape[0]}')

주문에 들어가있지 않은 주문 품목 수 : 4780
결측치 제거 후 총 주문품목수 : 495100


In [180]:
# 순매출을 구하기 위해서 prod와 병합(price 제외)
mixed = mix_ooi.merge(prod[['product_id','product_name','category','cost']],how='left',on='product_id',validate="m:1")
mixed.info()

<class 'pandas.DataFrame'>
RangeIndex: 495100 entries, 0 to 495099
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   495100 non-null  int64         
 1   order_id        495100 non-null  int64         
 2   product_id      495100 non-null  int64         
 3   quantity        495100 non-null  int64         
 4   unit_price      494862 non-null  float64       
 5   discount        495100 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
 10  product_name    487853 non-null  str           
 11  category        487853 non-null  str           
 12  cost            487853 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(4), str(4)
memory usage: 66.3 MB


In [181]:
cust_ids = set(cust['customer_id'])
order_ids = set(mixed['customer_id'])

print('고객 원장 고객 수:', len(cust_ids))
print('주문 데이터 고객 수:', len(order_ids))
print('주문에만 있는 고객 수:', len(order_ids - cust_ids))
print('고객 원장에만 있는 고객 수:', len(cust_ids - order_ids))

# 고객 원장과 연결되지 않는 주문 고객 908명은 고객 식별의 신뢰성을 확보하기 위해 분석에서 제외하였다.

고객 원장 고객 수: 4950
주문 데이터 고객 수: 5858
주문에만 있는 고객 수: 908
고객 원장에만 있는 고객 수: 0


In [182]:
# cust 병합(cust위주)
mixed= cust.merge(mixed,how="left",on="customer_id",validate="1:m")
mixed.info()
mixed['customer_id'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 492642 entries, 0 to 492641
Data columns (total 19 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   customer_id     492642 non-null  int64         
 1   name            492642 non-null  str           
 2   gender          457770 non-null  str           
 3   birth_date      464088 non-null  datetime64[us]
 4   signup_date     492642 non-null  datetime64[us]
 5   city            492642 non-null  str           
 6   email           477434 non-null  str           
 7   order_item_id   492642 non-null  int64         
 8   order_id        492642 non-null  int64         
 9   product_id      492642 non-null  int64         
 10  quantity        492642 non-null  int64         
 11  unit_price      492406 non-null  float64       
 12  discount        492642 non-null  float64       
 13  order_datetime  492642 non-null  datetime64[us]
 14  channel         492642 non-null  str           

4950

In [183]:
mixed['amount'] = (mixed['quantity'] * mixed['unit_price'])
mixed['p_amount'] = (mixed['quantity'] * mixed['unit_price'])*(1-mixed['discount'])
# 취소 환불 제거.
mixed = mixed[~(mixed['status'].isin(['canceled','returned']))]
mixed['status'].unique()
# 2024년 6월까지 데이터 제거
mixed=mixed[mixed['order_datetime']<'2024-07-01']

In [184]:
mixed=mixed.set_index('customer_id')

In [185]:
order_level = mixed.sort_values(['customer_id','order_datetime']).drop_duplicates(
        subset=['order_id']
    ).copy()

In [186]:
order_level['order_no'] = (
    order_level.groupby('customer_id')
         .cumcount()
)

order_level.info()

<class 'pandas.DataFrame'>
Index: 153951 entries, 1000 to 5949
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   name            153951 non-null  str           
 1   gender          143084 non-null  str           
 2   birth_date      144936 non-null  datetime64[us]
 3   signup_date     153951 non-null  datetime64[us]
 4   city            153951 non-null  str           
 5   email           149256 non-null  str           
 6   order_item_id   153951 non-null  int64         
 7   order_id        153951 non-null  int64         
 8   product_id      153951 non-null  int64         
 9   quantity        153951 non-null  int64         
 10  unit_price      153877 non-null  float64       
 11  discount        153951 non-null  float64       
 12  order_datetime  153951 non-null  datetime64[us]
 13  channel         153951 non-null  str           
 14  status          153951 non-null  str           
 15

In [187]:
# 새로 시리즈 구성(첫주문, 둘째주문). 이후 머지. 

first = order_level.loc[
    order_level['order_no']==0,
['order_datetime','order_id','channel','status','discount','p_amount','category']
].copy()

second = order_level.loc[
    order_level['order_no']==1, ['order_id','order_datetime',]
].copy()

member = first.merge(
    second,
    on='customer_id',
    how='left',
    suffixes=('_1','_2')
)

member.head()

,order_datetime_1,order_id_1,channel,status,discount,p_amount,category,order_id_2,order_datetime_2
customer_id,,,,,,,,,
1000,2024-01-30 08:13:14,22744,web,shipped,0.07,94395.0,도서,131547,2024-02-01 10:46:11
1001,2024-01-18 17:43:25,54658,app,delivered,0.23,9856.0,도서,48143,2024-02-04 15:27:45
1002,2024-01-06 10:37:15,144133,app,delivered,0.27,416392.0,전자,19136,2024-01-07 01:13:52
1003,2024-01-03 09:42:56,35039,web,delivered,0.06,38070.0,도서,34830,2024-01-04 11:12:53
1004,2024-01-01 04:23:36,107303,web,delivered,0.48,15600.0,도서,155255,2024-01-08 18:54:54


In [188]:
member['days'] = (
    member['order_datetime_2']
    - member['order_datetime_1']
).dt.days

In [189]:
member['success'] = (
    member['days']
    .le(30)
    .fillna(False)
)

In [190]:
# 6월은 포함하지 않음. (30일이 안됐기 때문에)
member = member.loc[
    member['order_datetime_1'].dt.month<=5
]

In [191]:
# 30일 이내 재구매율
member['success'].mean()

np.float64(0.8235650767987065)

In [192]:
(member['order_id_1'] == member['order_id_2']).sum()

np.int64(0)

In [193]:
print(f'30일 이내 2회차 전환율: {member["success"].mean():.2%}')

30일 이내 2회차 전환율: 82.36%


In [194]:
first_order_ids = member['order_id_1'].unique()

first_items = mixed.loc[
    mixed['order_id'].isin(first_order_ids)
].copy()
first_items = first_items.reset_index()

first_items[
    ['customer_id','order_id', 'category', 'discount']
].head()

,customer_id,order_id,category,discount
0,3293,110207,전자,0.43
1,3293,110207,뷰티,0.32
2,3293,110207,식품,0.13
3,1862,80151,의류,0.26
4,4955,138605,도서,0.33


In [195]:
first_items['sale_discount'] = (
    first_items['discount'] > 0
)

In [196]:
discount_by_order = (
    first_items.groupby('order_id')['sale_discount']
    .any()
    .reset_index()
)

In [197]:
member = member.merge(
    discount_by_order,
    left_on='order_id_1',
    right_on='order_id',
    how='left'
)

In [198]:
member = member.drop(columns='order_id')

In [199]:
member.groupby('sale_discount')['success'].agg(
    customer_count='count',
    converted_count='sum',
    conversion_rate='mean'
)
member

,order_datetime_1,order_id_1,channel,status,discount,p_amount,category,order_id_2,order_datetime_2,days,success,sale_discount
0,2024-01-30 08:13:14,22744,web,shipped,0.07,94395.0,도서,131547,2024-02-01 10:46:11,2,True,True
1,2024-01-18 17:43:25,54658,app,delivered,0.23,9856.0,도서,48143,2024-02-04 15:27:45,16,True,True
2,2024-01-06 10:37:15,144133,app,delivered,0.27,416392.0,전자,19136,2024-01-07 01:13:52,0,True,True
3,2024-01-03 09:42:56,35039,web,delivered,0.06,38070.0,도서,34830,2024-01-04 11:12:53,1,True,True
4,2024-01-01 04:23:36,107303,web,delivered,0.48,15600.0,도서,155255,2024-01-08 18:54:54,7,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
4943,2024-01-14 06:07:34,130316,store,paid,0.19,10935.0,도서,199779,2024-01-22 15:48:15,8,True,True
4944,2024-01-11 13:56:13,140414,app,shipped,0.41,16638.0,뷰티,145013,2024-01-24 07:21:05,12,True,True
4945,2024-01-03 20:08:15,185676,web,delivered,0.08,94300.0,NaN,168230,2024-01-10 11:30:39,6,True,True
4946,2024-02-11 21:29:18,63884,web,delivered,0.31,373980.0,전자,67453,2024-03-13 06:38:02,30,True,True


In [200]:
discount_result = (
    member.groupby('sale_discount')['success']
    .agg(
        customer_count='count',
        converted_count='sum',
        conversion_rate='mean'
    )
    .reset_index()
)

discount_result['conversion_rate'] = (
    discount_result['conversion_rate'] * 100
).round(2)

discount_result

,sale_discount,customer_count,converted_count,conversion_rate
0,False,13,8,61.54
1,True,4935,4067,82.41


In [201]:
category_by_order = (
    first_items[
        ['order_id', 'category']
    ]
    .drop_duplicates()
)
category_by_order

,order_id,category
0,110207,전자
1,110207,뷰티
2,110207,식품
3,80151,의류
4,138605,도서
...,...,...
13322,155281,전자
13323,155281,뷰티
13324,155281,가구
13326,63807,도서


In [202]:
analysis = category_by_order.merge(
    member[['channel','order_id_1','order_datetime_1','order_id_2','order_datetime_2','days','success','sale_discount',]],
    left_on='order_id',
    right_on='order_id_1',
    how='left'
)
analysis

,order_id,category,channel,order_id_1,order_datetime_1,order_id_2,order_datetime_2,days,success,sale_discount
0,110207,전자,app,110207,2024-01-05 08:54:22,38760,2024-03-09 06:05:27,63,False,True
1,110207,뷰티,app,110207,2024-01-05 08:54:22,38760,2024-03-09 06:05:27,63,False,True
2,110207,식품,app,110207,2024-01-05 08:54:22,38760,2024-03-09 06:05:27,63,False,True
3,80151,의류,app,80151,2024-01-28 23:17:32,162467,2024-02-04 23:50:14,7,True,True
4,138605,도서,store,138605,2024-01-27 08:34:44,16258,2024-01-28 14:15:59,1,True,True
...,...,...,...,...,...,...,...,...,...,...
10757,155281,전자,web,155281,2024-01-04 20:21:10,132552,2024-01-29 00:20:19,24,True,True
10758,155281,뷰티,web,155281,2024-01-04 20:21:10,132552,2024-01-29 00:20:19,24,True,True
10759,155281,가구,web,155281,2024-01-04 20:21:10,132552,2024-01-29 00:20:19,24,True,True
10760,63807,도서,store,63807,2024-02-25 20:08:32,2163,2024-02-29 07:43:43,3,True,True


In [203]:
analysis = analysis.drop(columns='order_id')

In [204]:
# 요건에서 처음 산 물품의 카테고리를 중요하게 다루므로 category_x만 표기함.
conversion_table = (
    analysis.groupby(
        ['channel', 'category', 'sale_discount']
    )['success']
    .agg(
        customer_count='count',
        converted_count='sum',
        conversion_rate='mean'
    )
    .reset_index()
)
conversion_table['conversion_rate'] = (
    conversion_table['conversion_rate'] * 100
).round(2)

conversion_table = conversion_table.sort_values(
    ['conversion_rate', 'customer_count'],
    ascending=[True, False]
)

conversion_table.sort_values(['conversion_rate'],ascending=False)

,channel,category,sale_discount,customer_count,converted_count,conversion_rate
2,app,도서,False,2,2,100.00
15,store,의류,False,1,1,100.00
22,web,뷰티,False,1,1,100.00
11,store,도서,False,1,1,100.00
8,app,전자,False,1,1,100.00
4,app,뷰티,False,1,1,100.00
9,app,전자,True,671,572,85.25
5,app,뷰티,True,423,359,84.87
3,app,도서,True,786,665,84.61
1,app,가구,True,431,363,84.22


### 결과 분석

구매 인원이 5명 이하인 조합을 제외하면, 재구매율이 가장 낮은 조합은 스토어에서 가구를 구매한 고객이었다. 그다음으로는 웹에서 도서를 구매한 고객, 웹에서 의류를 구매한 고객 순으로 나타났다.

할인을 받지 않은 고객의 재구매율은 상대적으로 낮게 나타났지만, 할인 없이 구매한 고객은 총 21명으로 표본이 매우 적어 일반적인 경향으로 해석하기에는 한계가 있다.

#### 가설

스토어에서 구매한 가구의 재구매율이 낮은 이유는 가구의 구매 주기가 길고, 한 번 구매하면 단기간 내 재구매가 필요한 품목이 아니기 때문일 가능성이 있다. 또한 부피와 무게가 큰 상품 특성상 오프라인 방문 구매 빈도가 낮을 수도 있다.

웹에서 도서와 의류의 재구매율이 낮게 나타난 원인은 현재 데이터만으로는 확인할 수 없다. 다만 상품 만족도, 가격 경쟁, 배송 경험 등의 요인이 영향을 미쳤을 가능성이 있으므로 추가적인 고객 데이터와 함께 분석할 필요가 있다.

개선 방안으로는 스토어 가구 판매를 즉시 축소하기보다는 온라인 채널 중심의 프로모션과 배송 서비스를 강화하여 효과를 비교하는 것이 바람직하다. 또한 웹 구매 고객의 재구매를 유도하기 위해 재구매 할인 쿠폰을 제공하고, 의류의 경우 AI 기반 가상 피팅 서비스를 도입하여 구매 만족도를 높이는 방안을 고려할 수 있다.

In [205]:
discount_table = (
    member.groupby('sale_discount')['success']
    .agg(
        첫구매='count',
        삼십일내재구매='sum',
        재구매율='mean'
    )
)

discount_table['재구매율'] = (
    discount_table['재구매율'] * 100
).round(2)

discount_table

,첫구매,삼십일내재구매,재구매율
sale_discount,,,
False,13,8,61.54
True,4935,4067,82.41


In [206]:
#카테고리별 구매 전환율
cate_table=conversion_table.groupby('category').agg(
    첫구매=("customer_count","sum"),
    삼십일내재구매=("converted_count","sum")
)
cate_table["재구매율"]=round(cate_table["삼십일내재구매"]/cate_table["첫구매"]*100,2)
cate_table

,첫구매,삼십일내재구매,재구매율
category,,,
가구,1357,1105,81.43
도서,2669,2206,82.65
뷰티,1437,1189,82.74
식품,1481,1233,83.25
의류,1363,1116,81.88
전자,2256,1872,82.98


In [207]:
channel_table = (
    member.groupby('channel')['success']
    .agg(
        첫구매='count',
        삼십일내재구매='sum',
        재구매율='mean'
    )
)

channel_table['재구매율'] = (
    channel_table['재구매율'] * 100
).round(2)

channel_table

,첫구매,삼십일내재구매,재구매율
channel,,,
app,1468,1227,83.58
store,726,589,81.13
web,2754,2259,82.03
